In [1]:
%load_ext cudf.pandas

In [2]:
# import dias.rewriter
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
# %%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
# from tpch import load_lineitem, load_part, load_nation, load_partsupp, load_supplier, q20
import pandas as pd

DATA_ROOT = "/home/colinc/code/tpch/data/tpch_15"  #"/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}



In [4]:
### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # Read with cudf-backed pandas and parse date columns directly on the GPU
    df = pd.read_csv(
        data_path,
        sep="|",
        parse_dates=["L_SHIPDATE", "L_RECEIPTDATE", "L_COMMITDATE"],
        **storage_options
    )
    print(df.columns)
    return df

# Load into GPU-backed DataFrame
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [5]:
### cell 1 ###
def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = f"{data_folder}/part.tbl"
    # Use read_csv (GPU‐accelerated via cudf.pandas) since the file is a delimited text file
    df = pd.read_csv(
        data_path,
        sep="|"
    )
    return df

part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [6]:
### cell 2 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)


In [7]:
### cell 3 ###

def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")  
    return df

partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)


In [8]:
### cell 4 ###

def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:

### cell 5 ###
# Filter predicates using GPU‐friendly operations (no pd.Timestamp scalars)
psel = part.P_NAME.str.startswith("azure")
nsel = nation.N_NAME == "JORDAN"
lsel = (
    lineitem.L_SHIPDATE >= "1996-01-01"
) & (
    lineitem.L_SHIPDATE < "1997-01-01"
)

fpart = part[psel]
fnation = nation[nsel]
flineitem = lineitem[lsel]

jn1 = fpart.merge(partsupp, left_on="P_PARTKEY", right_on="PS_PARTKEY")
jn2 = jn1.merge(
    flineitem,
    left_on=["PS_PARTKEY", "PS_SUPPKEY"],
    right_on=["L_PARTKEY", "L_SUPPKEY"],
)

gb = jn2.groupby(
    ["PS_PARTKEY", "PS_SUPPKEY", "PS_AVAILQTY"],
    as_index=False,
    sort=False
)["L_QUANTITY"].sum()

# retain only those with available quantity > half of shipped quantity
gbsel = gb.PS_AVAILQTY > (0.5 * gb.L_QUANTITY)
fgb = gb[gbsel]

jn3 = fgb.merge(supplier, left_on="PS_SUPPKEY", right_on="S_SUPPKEY")
jn4 = fnation.merge(jn3, left_on="N_NATIONKEY", right_on="S_NATIONKEY")[["S_NAME", "S_ADDRESS"]]

total = jn4.sort_values("S_NAME").drop_duplicates()